In [ ]:
!pip install datasets
from datasets import load_dataset

# Load the dataset
dataset_dict = load_dataset("cnn_dailymail", "3.0.0")

# Convert the 'train', 'validation', and 'test' splits to pandas DataFrames
train_df = dataset_dict['train'].to_pandas()
val_df = dataset_dict['validation'].to_pandas()
test_df = dataset_dict['test'].to_pandas()

# Select the first 100 instances from each split (optional)
train_df = dataset_dict['train'].select(range(1000))
val_df =  dataset_dict['validation'].select(range(200))
test_df =  dataset_dict['test'].select(range(200))

# Print the shapes of the DataFrames
print("Training data shape:", train_df.shape)
print("Validation data shape:", val_df.shape)
print("Test data shape:", test_df.shape)



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Training data shape: (1000, 3)
Validation data shape: (200, 3)
Test data shape: (200, 3)


In [ ]:
!pip install keras tensorflow
import tensorflow as tf
from tensorflow.keras.layers import Layer, Embedding, Bidirectional, LSTM, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

In [ ]:
class WordAttention(Layer):
    def __init__(self, **kwargs):
        super(WordAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="attention_weight",
                                 shape=(input_shape[-1], input_shape[-1]),
                                 initializer="random_normal",
                                 trainable=True)
        self.b = self.add_weight(name="attention_bias",
                                 shape=(input_shape[-1],),
                                 initializer="zeros",
                                 trainable=True)
        self.u = self.add_weight(name="attention_context_vector",
                                 shape=(input_shape[-1], 1),
                                 initializer="random_normal",
                                 trainable=True)
        super(WordAttention, self).build(input_shape)

    def call(self, x):
        x = tf.cast(x, tf.float32)
        print("Input shape to WordAttention:", x.shape)

        # Get dynamic shape values
        batch_size = tf.shape(x)[0]
        num_tokens = tf.shape(x)[1]  # Get num_tokens dynamically from input shape
        feature_dim = x.shape[-1]  # Use the last dimension as feature_dim

        #Calculate uit
        uit = K.tanh(K.dot(x, self.W) + self.b)
        print("uit shape:", uit.shape)

        #Calculate ait
        ait = K.dot(uit, self.u)
        print("ait shape:", ait.shape)

        #Apply softmax to get attention weights
        ait = tf.nn.softmax(ait, axis=1)
        print("ait shape after softmax:", ait.shape)

        #Perform element-wise multiplication
        weighted_input = x * ait # (batch_size, num_tokens, feature_dim)
        print("weighted_input shape:", weighted_input.shape)

        #Sum weighted inputs to get context vector
        output = K.sum(weighted_input, axis=1) # (batch_size, feature_dim)
        print("output shape:", output.shape)

        return output

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

In [ ]:
class SentenceAttention(tf.keras.layers.Layer):
    def __init__(self, attention_dim):
        super(SentenceAttention, self).__init__()
        self.attention_dim = attention_dim
        self.W = tf.keras.layers.Dense(attention_dim)
        self.u = tf.keras.layers.Dense(1)

    def call(self, x):
        # x shape: (batch_size, sentence_count, embedding_dim)

        # Generate uit
        uit = self.W(x)  # (batch_size, sentence_count, attention_dim)

        # Generate attention weights
        ait = self.u(uit)  # (batch_size, sentence_count, 1)
        ait = tf.nn.softmax(ait, axis=1)

        # Ensure proper broadcasting
        weighted_input = x * ait  # Broadcasting should work now

        # Sum over the sentence dimension
        output = tf.reduce_sum(weighted_input, axis=1)  # (batch_size, embedding_dim)

        return output

In [ ]:
def call(self, x):
    print(f"SentenceAttention input shape: {x.shape}")
    uit = self.W(x)
    print(f"uit shape: {uit.shape}")
    ait = self.u(uit)
    print(f"ait shape before softmax: {ait.shape}")
    ait = tf.nn.softmax(ait, axis=1)
    print(f"ait shape after softmax: {ait.shape}")
    weighted_input = x * ait
    print(f"weighted_input shape: {weighted_input.shape}")
    output = tf.reduce_sum(weighted_input, axis=1)
    print(f"output shape: {output.shape}")
    return output

In [ ]:
from datasets import DatasetDict

# Define dataset_dict using DatasetDict
dataset_dict = DatasetDict({
    "train": train_df,
    "validation": val_df,
    "test": test_df
})

# Verify the dataset
print(dataset_dict)


DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 200
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 200
    })
})


In [ ]:
from transformers import PegasusTokenizer, PegasusForConditionalGeneration

# Define model name
model_name = "google/pegasus-cnn_dailymail"

# Load the tokenizer and model
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM
from tensorflow.keras.models import Model,Sequential
def hierarchical_attention_model(vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix):
    sentence_input = Input(shape=(sentence_count, word_count))

    # Word encoder
    word_encoder = Sequential([
        Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=False),
        Bidirectional(LSTM(100, return_sequences=True)),
        WordAttention()
    ])

    # Apply word encoder to each sentence
    sentence_encoder = tf.keras.layers.TimeDistributed(word_encoder)(sentence_input)
    sentence_bi_lstm = Bidirectional(LSTM(100, return_sequences=True))(sentence_encoder)
    # Pass attention_dim when creating SentenceAttention
    sentence_attention = SentenceAttention(attention_dim=200)(sentence_bi_lstm)

    return tf.keras.models.Model(inputs=sentence_input, outputs=sentence_attention)

In [ ]:
import numpy as np
vocab_size = 10000
embedding_dim = 300
sentence_count = 10
word_count = 20
embedding_matrix = np.random.rand(vocab_size, embedding_dim)

hierarchical_model = hierarchical_attention_model(vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix)

In [ ]:
hierarchical_model.compile()

The WordAttention layer is missing from the model summary because it's inside the word_encoder, which is wrapped in TimeDistributed. Here's why it's not explicitly listed:

In [ ]:
hierarchical_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 10, 20)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed (TimeDistributed)   │ (None, 10, 200)             │       3,361,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_1 (Bidirectional)      │ (None, 10, 200)             │         240,800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sentence_attention                   │ (None, 200)                 │          40,401 │
│ (SentenceAttention)                  │                             │                 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,642,401 (13.89 MB)

 Trainable params: 642,401 (2.45 MB)

 Non-trainable params: 3,000,000 (11.44 MB)

In [ ]:
import torch

In [ ]:
def HTA(batch, tokenizer, sentence_count, word_count):
    # Tokenize input articles
    inputs = tokenizer(
        batch["article"],
        truncation=True,
        padding="max_length",
        max_length=min(512, sentence_count * word_count),
        return_tensors="np"
    )

    # # Reshape input to fit HAN
    # input_ids = np.array(inputs["input_ids"]).reshape(-1, sentence_count, word_count)

    # # Convert token IDs to embeddings using HAN (Optional if you're not using embeddings directly)
    # hta_output = hierarchical_model(input_ids)

    # # Tokenize target summaries (highlights)
    targets = tokenizer(
        batch["highlights"],
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    return {
        "input_ids": inputs["input_ids"],
        "labels": targets["input_ids"]
    }



In [ ]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

def prepare_dataset(dataset_dict, vocab_size, embedding_dim, sentence_count, word_count, tokenizer, embedding_matrix, hierarchical_attention_model):
    # Define preprocessing function with fixed parameters
    hta_model_instance = hierarchical_attention_model(vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix)
    def preprocess_with_fixed_params(batch):
        return HTA(
            batch,
            tokenizer,
            sentence_count,
            word_count
        )


    processed_datasets = {}
    for split in dataset_dict.keys():
        processed_datasets[split] = dataset_dict[split].map(
            preprocess_with_fixed_params,
            batched=True,
            batch_size=8,
            remove_columns=dataset_dict[split].column_names,
            desc=f"Processing {split} split"
        )

    return processed_datasets

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
from transformers import PegasusTokenizer
processed_dataset_dict = prepare_dataset(
    dataset_dict=dataset_dict,
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    sentence_count=sentence_count,D
    word_count=word_count,
    tokenizer=tokenizer,
    embedding_matrix=embedding_matrix,
    hierarchical_attention_model=hierarchical_attention_model
)

Processing train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Processing validation split:   0%|          | 0/200 [00:00<?, ? examples/s]

Processing test split:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
print(processed_dataset_dict["train"])
print(processed_dataset_dict["validation"])
print(processed_dataset_dict["test"])

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 1000
})
Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 200
})
Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 200
})


In [ ]:
print(processed_dataset_dict)

{'train': Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 1000
}), 'validation': Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 200
}), 'test': Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 200
})}


In [ ]:
from transformers import PegasusTokenizer, PegasusForConditionalGeneration, Trainer, TrainingArguments, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./pegasus-finetuned",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=7,
    weight_decay=0.001,
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=500,
    load_best_model_at_end=True,
    predict_with_generate=True,
    remove_unused_columns=False,
    fp16 = True,
)


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset_dict["train"],
    eval_dataset=processed_dataset_dict["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

<ipython-input-125-0ef18ffda59f>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.718991


In [ ]:
def evaluate_summaries(dataset, model, tokenizer):
    references = []
    predictions = []

    for sample in dataset:
        input_text = f"{sample['article']} {sample['highlights']}"
        reference_summary = sample['highlights']
        references.append(reference_summary)

        # Generate summary
        inputs = tokenizer(input_text, max_length=1024, truncation=True, return_tensors="pt").to(model.device)
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=128,
            num_beams=4,
            early_stopping=True
        )
        generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        predictions.append(generated_summary)

    return references, predictions


In [ ]:
test_references, test_predictions = evaluate_summaries(test_df, model, tokenizer)


In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score

def calculate_average_precision(test_references, test_predictions):
    # Create a set of all unique words across human and predicted summaries
    all_words = set(word for summary in test_references for word in summary.lower().split())
                set(word for summary in test_predictions for word in summary.lower().split())

    # Convert human summaries and predicted summaries into binary vectors
    human_vectors = [
        np.array([1 if word in summary.lower().split() else 0 for word in all_words])
        for summary in test_references
    ]
    predicted_vectors = [
        np.array([1 if word in summary.lower().split() else 0 for word in all_words])
        for summary in test_predictions
    ]

    # Calculate the average precision score
    ap_scores = []
    for true, pred in zip(human_vectors, predicted_vectors):
        try:
            ap_scores.append(average_precision_score(true, pred))
        except ValueError:
            ap_scores.append(0.0)

    return np.mean(ap_scores)

In [ ]:
ap_score = calculate_average_precision(test_references, test_predictions)
print(f"Average Precision Score: {ap_score}")

In [ ]:
!pip install evaluate
!pip install rouge_score
import evaluate
rouge = evaluate.load("rouge")

In [ ]:

results = rouge.compute(predictions=test_predictions, references=test_references)

In [ ]:
print(f"ROUGE-1: {results['rouge1']}")
print(f"ROUGE-2: {results['rouge2']}")
print(f"ROUGE-L: {results['rougeL']}")

In [ ]:
index = 55
print("Reference Summary:")
print(test_references[index])
print("\nPredicted Summary:")
print(test_predictions[index])

In [ ]:
import pandas as pd

# Create a DataFrame
df_results = pd.DataFrame({
    "Reference Summary": test_references,
    "Predicted Summary": test_predictions
})

# Save to CSV file
df_results.to_csv("summary_comparison.csv", index=False)

print("CSV file saved successfully!")


In [ ]:
from nltk.translate.bleu_score import corpus_bleu

# Tokenize references and predictions
tokenized_references = [[ref.split()] for ref in test_references]  # BLEU expects a list of lists
tokenized_predictions = [pred.split() for pred in test_predictions]

# Compute BLEU Score
bleu_score = corpus_bleu(tokenized_references, tokenized_predictions)

print(f"BLEU Score: {bleu_score:.4f}")


In [ ]:
import nltk
nltk.download('punkt')

from nltk.translate.bleu_score import sentence_bleu, corpus_bleu

# Compute BLEU for each sentence
for i in range(100):  # Print BLEU scores for the first 5 samples
    reference = [test_references[i].split()]  # BLEU expects a list of lists
    prediction = test_predictions[i].split()

    score = sentence_bleu(reference, prediction)
    print(f"Sample {i+1} - BLEU Score: {score:.4f}")


In [ ]:
!pip install bert_score
from bert_score import score

# Assuming test_predictions contains the generated summaries
generated_summaries = test_predictions

# Assuming test_references contains the actual summaries
actual_summaries = test_references

P, R, F1 = score(generated_summaries, actual_summaries, lang="en", model_type="roberta-large")

avg_precision = P.mean().item()
bert_f1_score = F1.mean().item()

print(f"\n🔹 BERTScore (Improved): {bert_f1_score:.4f}")
print(f" Average Precision (BERTScore P): {avg_precision:.4f}")